In [1]:
import datetime as dt
from datetime import datetime, timedelta
import dask
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt

import xarray as xr
from glob import glob
from time import time
import warnings

import pandas as pd

warnings.simplefilter("ignore")

# Set Parameters

# Plots
- Maps
    - [ ] Monats Aufenthaltswahrscheinlichkeiten
    - [ ] Jahres Aufenthaltswahrscheinlichkeiten
    - [ ] Linen zwischen Stationen
- Connectivitätsmatrix
    - [ ] Monats Averages (logscale)
    - [ ] Jahres Averages (logscale)
    - [ ] Einzelne Jahre

In [2]:
# Parameters
site_counter = np.arange(16)
years = np.arange(2016,2025)
year = 2024
nbins = 100
isPapermill = True
start_site_counter = 15
samples = [
    "Speicherkoog", "KB03", "SW08",
    "Wilhelmshaven", "GB96", "H21",
    "Boknis_Eck", "Gdynia", "Riga",
    "IU7_c", "LL17_c", "Finland",
    "BB23", "BB36", "Speicherkoog_Außen",
    "Wilhelmshaven_Außen",
]
lats = [
    54.0929, 54.692, 54.413,
    53.513, 56.54, 54.666,
    54.5169, 54.5829, 57.3911,
    59.489, 59.02, 59.77,
    55.1751, 54.575, 54.11,
    53.637,
]
lons = [
    8.9487, 10.1035, 11.617,
    8.149, 19.5809, 13.0224,
    10.034, 18.6042, 23.8588,
    21.2021, 21.0478, 23.2663,
    15.4352, 16.001, 8.5,
    8.15,
]

number_sites = len(samples)
number_years = len(years)
dist = 20

# Functions

In [3]:
def connectivity_function(
    ds=None,
    dist=None,
):
    # Mean radius of the Earth in km
    R = 6371  

    # Stations (targets)
    obs_lon_deg = ds.lon
    obs_lat_deg = ds.lat
    obs_lon_km = R * np.cos(np.radians(obs_lat_deg)) * np.radians(obs_lon_deg)
    obs_lat_km = R * np.radians(obs_lat_deg)

    # Particles
    target_lon_deg = xr.DataArray(lons)
    target_lat_deg = xr.DataArray(lats)
    target_lon_km = R * np.cos(np.radians(target_lat_deg)) * np.radians(target_lon_deg)
    target_lat_km = R * np.radians(target_lat_deg)

    # Calculate distance to target
    distance_to_target = (
        ((obs_lon_km - target_lon_km) ** 2) + ((obs_lat_km - target_lat_km) ** 2)
    ) ** 0.5

    # Return number of particles within a certain distance from target
    return (distance_to_target <= dist).sum(dim=["obs", "trajectory"]).compute().values
    # return distance_to_target

In [4]:
read_path = "/gxfs_work/geomar/smomw597/2025_copepod_dispersal/output/"
out_path_matrix = "/gxfs_work/geomar/smomw597/2025_copepod_dispersal/output/Connectivity/"
out_path_hist = "/gxfs_work/geomar/smomw597/2025_copepod_dispersal/output/hist/"

In [5]:
file = Path(
    read_path +
    f"PPmill_Nested_{year}0601-{year}1101_dt15min_site{start_site_counter:02d}_d0m-25m_N1000_seed123.zarr"
)

In [6]:
ds_trajectories = xr.Dataset()
monthly_connectivity_matrices = {}
export_vars = ["lat", "lon", "time", "S", "T", "eta", "z", "h0", "age_sec"]
export_vars = ["lat", "lon"]
if isPapermill == False:
    month_split = [None, 4,9,13,17,22]
else:
    month_split = [None, 30000,61000,92000,122000,153000]
expand_dims_dict = {"site":[start_site_counter], "year":[year],}

In [7]:
print("get file", year, start_site_counter)
if isPapermill == False:
    ds_trajectories = (
        xr.open_zarr(file)
        .isel(trajectory=slice(None, None, 7000))
        .chunk(chunks={'trajectory':-1,'obs':-1,})
    )
else:
    ds_trajectories = (
        xr.open_zarr(file)
        .chunk(chunks={'trajectory':100000,'obs':100,})
    )

get file 2024 15


In [8]:
time_mask = ((ds_trajectories.age_sec/(60*60*24)) > 10).compute()
ds_positions = ds_trajectories.get(["lat", "lon", "age_sec", "time"])
ds_positions = ds_positions.compute()

In [9]:
# for month_counter, month in enumerate(range(6,11)):
#     connectivity_matrix = np.zeros((number_sites, number_sites), dtype=int)
#     ds_month_positions = ds_positions.isel(
#         trajectory = slice(month_split[month_counter], month_split[month_counter+1])
#     )
#     connectivity_matrix[start_site_counter,:] = connectivity_function(ds=ds_month_positions, dist=dist)
#     monthly_connectivity_matrices[year, month] = connectivity_matrix

In [10]:
# np.save(Path(out_path_matrix+f"connectivity_matrices_{year}_s{start_site_counter:02d}.npy"), monthly_connectivity_matrices)

In [11]:
# ds_variables = ds_trajectories.get(export_vars).expand_dims(expand_dims_dict)
# ds_variables["depth_m"] = (
#     (ds_variables.eta - ds_variables.z * (ds_variables.eta + ds_variables.h0))
# ).compute()
# ds_variables = ds_variables.drop_vars(["eta", "z", "h0"])

In [12]:
lonbins = np.linspace(0,30,nbins+1)
latbins = np.linspace(50,65,nbins+1)

In [13]:
# Group by months and age
# ds_positions["age_day"] = ds_positions.age_sec/(60*60*24)


In [14]:
# ds_age = ds_positions.groupby_bins("age_day", np.arange(0,29), labels=np.arange(1,29))

In [ ]:
ds = ds_positions.drop_vars("time").groupby_bins(
    "age_sec", 
    np.arange(0,29*(60*60*24),(60*60*24)),
    labels=np.arange(1,29)
)
hist_age = {}
for age in ds:
    test_lons_array = age[1].lon.values
    test_lats_array = age[1].lat.values
    hist, xedges, yedges = np.histogram2d(
        test_lons_array, test_lats_array,
        bins=(lonbins, latbins),
    )
    hist_age[age[0]] = hist
np.save(Path(out_path_hist+f"age/hist2d_age_n{nbins}_{year}_s{start_site_counter:02d}.npy"), hist_age)

In [17]:
ds_positions = ds_positions.drop_vars("age_sec").where(time_mask, drop = True)

In [18]:
ds = ds_positions.where(time_mask, drop = True).groupby("time.month")
print("group")
hist_monthly = {}
for m in ds:
    test_lons_array = m[1].lon.values
    test_lats_array = m[1].lat.values
    hist, xedges, yedges = np.histogram2d(
        test_lons_array, test_lats_array,
        bins=(lonbins, latbins),
    )
    hist_monthly[m[0]] = hist
    print(m[0])
np.save(Path(out_path_hist+f"monthly/hist2d_monthly_n{nbins}_{year}_s{start_site_counter:02d}.npy"), hist_monthly)

group
6.0
7.0
8.0
9.0
10.0


In [19]:
test_lons = ds_positions.lon
test_lats = ds_positions.lat

test_lons_array = test_lons.stack(z=test_lons.dims).dropna(dim="z").values
test_lats_array = test_lats.stack(z=test_lats.dims).dropna(dim="z").values
hist, xedges, yedges = np.histogram2d(
    test_lons_array, test_lats_array,
    bins=(lonbins, latbins),
)
print(hist.nbytes/1e9)
np.save(Path(out_path_hist+f"total/hist2d_n{nbins}_{year}_s{start_site_counter:02d}.npy"), hist)

8e-05


In [20]:
# ds_variables.to_netcdf(
#     path=Path(out_path+f"ds_variables_{year}_s{start_site_counter:02d}.nc"),
#     mode="a",
# )

In [21]:
# isTest = False
# ds_trajectories = xr.Dataset()
# connectivity_matrix_monthly = {}
# export_vars = ["lat", "lon", "time", "S", "T", "eta", "z", "h0", "age_sec"]
# if isTest == True:
#     month_count = [None, 4,9,13,17,22]
#     years = np.arange(2016,2018)
# else:
#     month_count = [None, 30000,61000,92000,122000,153000]
#     years = np.arange(2016,2025)
# # connectivity_matrix = np.zeros((number_sites, number_sites), dtype=int)
# for year in years:
#     for start_site_counter in site_counter:
#         expand_dims_dict = {"site":[start_site_counter], "year":[year],}
#         file = get_filepath(year, start_site_counter)
#         print("get file", year, start_site_counter)
#         if isTest == True:
#             ds_trajectories = (
#                 xr.open_zarr(file)
#                 .isel(trajectory=slice(None, None, 7000))
#                 .chunk(chunks={'trajectory':-1,'obs':-1,})
#             )
#         else:
#             ds_trajectories = (
#                 xr.open_zarr(file)
#                 .chunk(chunks={'trajectory':100000,'obs':100,})
#             )
#         if start_site_counter == min(site_counter):
#             ds_variables_y = ds_trajectories.get(export_vars).expand_dims(expand_dims_dict)
#             time_mask = ((ds_trajectories.age_sec/(60*60*24)) > 10).compute()
#         else:
#             ds_var_temp = ds_trajectories.get(export_vars).expand_dims(expand_dims_dict)
#             ds_variables_y = xr.combine_by_coords([ds_variables_y,ds_var_temp])
#         ds_trajectories = ds_trajectories.where(time_mask, drop = True)
#         ds_positions = ds_trajectories.get(["lat", "lon"]).compute()
#         for month in np.arange(6,11):
#             temp_connectivity_matrix = np.zeros((number_sites, number_sites), dtype=int)
#             ds_month_positions = ds_positions.isel(trajectory = slice(month_count[month-6], month_count[month-5]))
#             temp_connectivity_matrix[start_site_counter,:]  = connectivity_function(ds=ds_month_positions, dist=dist)
#             if start_site_counter == min(site_counter):
#                 connectivity_matrix_monthly[year, month] = temp_connectivity_matrix
#             else:
#                 connectivity_matrix_monthly[year, month] = connectivity_matrix_monthly[year, month] + temp_connectivity_matrix
#     if year == min(years):
#         ds_variables = ds_variables_y
#     else:
#         ds_variables = xr.combine_by_coords([ds_variables,ds_variables_y])